# VLM Financial Table Extractor

Notebook ini mengekstrak tabel keuangan dari PDF menggunakan **Qwen2-VL-7B** (Vision Language Model).

**Alur kerja:**
1. Cek GPU
2. Install dependencies
3. Upload PDF
4. Definisi fungsi
5. Konfigurasi & jalankan
6. Download hasil

In [ ]:
import torch

print('GPU tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU       :', torch.cuda.get_device_name(0))
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'VRAM      : {vram} GB')
else:
    print()
    print('GPU tidak aktif! Aktifkan dulu:')
    print('  Runtime > Change runtime type > T4 GPU > Save')
    print('  Kemudian jalankan ulang semua cell.')

## Step 1 — Install Dependencies

Jalankan cell ini sekali. Akan memakan waktu 1-2 menit.

In [ ]:
!pip install -q pypdfium2 "transformers>=4.37" accelerate
print('Instalasi selesai!')

## Step 2 — Upload PDF

Klik **Choose Files** lalu pilih file PDF laporan keuangan dari komputer Anda.

In [ ]:
from google.colab import files

print('Pilih file PDF dari komputer Anda...')
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f'File terupload: {pdf_filename}')

## Step 3 — Definisi Fungsi VLM

Jalankan cell ini untuk memuat semua fungsi (output hanya satu baris konfirmasi).

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import pypdfium2 as pdfium
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration


SYSTEM_PROMPT = '''
You are an expert document structure extraction system.

Your task:
Extract a financial table from an image into STRICT JSON.

Important rules:
1. Preserve row hierarchy using indentation level.
2. Preserve the exact row order from top to bottom.
3. Extract account labels carefully.
4. Extract numeric values exactly as shown.
5. If a row has no value, return null.
6. If a row is a section/group header, still include it.
7. Negative values written in parentheses must stay as strings, e.g. "(7)".
8. Do not explain anything.
9. Output ONLY valid JSON.
10. Do not wrap JSON in markdown fences.

Return this schema exactly:

{
  "table_title": "string or null",
  "columns": ["account", "YEAR1", "YEAR2"],
  "rows": [
    {
      "level": 0,
      "account": "string",
      "YEAR1": "string or null",
      "YEAR2": "string or null",
      "row_type": "group|item|label_only"
    }
  ]
}

Replace YEAR1, YEAR2 with the actual year values found in the table header.
'''

USER_PROMPT = '''
Extract the table from this image.

Focus on:
- account names on the left
- value columns on the right
- indentation / hierarchy level
- section headers vs detail rows

Infer `level` like this:
- 0 = main section
- 1 = first indent
- 2 = deeper indent
- 3 = deeper indent if needed

Infer `row_type` like this:
- group = a grouping/header row
- item = a normal row with values
- label_only = a row without values that is not clearly a group header

Return STRICT JSON only.
'''


def pdf_to_images(pdf_path, start_page, end_page, dpi=150):
    doc = pdfium.PdfDocument(pdf_path)
    total = len(doc)
    start_idx = max(1, start_page) - 1
    end_idx = min(total, end_page) - 1
    scale = dpi / 72
    images = []
    for idx in range(start_idx, end_idx + 1):
        page = doc[idx]
        bitmap = page.render(scale=scale, rotation=0)
        images.append((idx + 1, bitmap.to_pil()))
    doc.close()
    return images


def get_total_pages(pdf_path):
    doc = pdfium.PdfDocument(pdf_path)
    total = len(doc)
    doc.close()
    return total


def extract_json(text):
    text = text.strip()
    text = re.sub(r'^```json\s*', '', text)
    text = re.sub(r'^```\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError('Model output tidak mengandung JSON yang valid.')
    return json.loads(text[start:end + 1])


def normalize_rows(data):
    rows = data.get('rows', [])
    columns = data.get('columns', [])
    value_cols = [c for c in columns if c != 'account']
    normalized = []
    for r in rows:
        row = {
            'level': int(r.get('level', 0)) if r.get('level') is not None else 0,
            'account': r.get('account'),
            'row_type': r.get('row_type', 'item'),
        }
        for col in value_cols:
            row[col] = r.get(col)
        normalized.append(row)
    return normalized


def build_parent_child(rows):
    stack = []
    output = []
    for row in rows:
        level = row['level']
        while stack and stack[-1]['level'] >= level:
            stack.pop()
        parent = stack[-1]['account'] if stack else None
        output.append({**row, 'parent_account': parent})
        stack.append(row)
    return output


def load_model(model_name):
    print('Memuat processor...')
    processor = AutoProcessor.from_pretrained(model_name)
    print('Memuat model (mungkin perlu beberapa menit)...')
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto'
    )
    print('Model siap!')
    return processor, model


def run_inference(image, processor, model):
    image = image.convert('RGB')
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': USER_PROMPT},
        ]},
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], padding=True, return_tensors='pt')
    inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            temperature=0.0,
        )
    output_text = processor.batch_decode(
        generated_ids[:, inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )[0]
    return output_text


def save_page_output(data, page_num, output_dir):
    rows = normalize_rows(data)
    rows = build_parent_child(rows)
    label = f'page_{page_num:03d}'
    json_path = output_dir / f'{label}.json'
    csv_path = output_dir / f'{label}.csv'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump({
            'page': page_num,
            'table_title': data.get('table_title'),
            'columns': data.get('columns', []),
            'rows': rows,
        }, f, ensure_ascii=False, indent=2)
    df = pd.DataFrame(rows)
    df.insert(0, 'page', page_num)
    df.to_csv(csv_path, index=False)
    return rows, data.get('table_title'), data.get('columns', [])


def save_combined_output(all_results, output_dir):
    json_path = output_dir / 'combined.json'
    csv_path = output_dir / 'combined.csv'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    dfs = []
    for result in all_results:
        if not result.get('rows'):
            continue
        df = pd.DataFrame(result['rows'])
        df.insert(0, 'page', result['page'])
        dfs.append(df)
    if dfs:
        pd.concat(dfs, ignore_index=True).to_csv(csv_path, index=False)
    return json_path, csv_path


print('Semua fungsi berhasil dimuat!')

## Step 4 — Konfigurasi & Jalankan

Ubah parameter di bawah sesuai kebutuhan, lalu jalankan kedua cell berikut.

In [ ]:
# ============================================================
# KONFIGURASI — ubah nilai di bawah sesuai kebutuhan
# ============================================================
PDF_PATH   = pdf_filename  # otomatis dari hasil upload di Step 2
START_PAGE = 1             # halaman awal (1-indexed)
END_PAGE   = None          # halaman akhir; None = semua halaman
DPI        = 150           # kualitas render (100-200 disarankan)
MODEL_NAME = 'Qwen/Qwen2-VL-7B-Instruct'
OUTPUT_DIR = 'vlm_output'

end_label = str(END_PAGE) if END_PAGE else 'akhir'
print(f'PDF      : {PDF_PATH}')
print(f'Halaman  : {START_PAGE} s/d {end_label}')
print(f'DPI      : {DPI}')
print(f'Model    : {MODEL_NAME}')
print(f'Output   : {OUTPUT_DIR}/')

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

total_pages = get_total_pages(PDF_PATH)
end_page = END_PAGE if END_PAGE else total_pages

print(f'Total halaman PDF : {total_pages}')
print(f'Halaman diproses  : {START_PAGE} s/d {end_page}')
print()

print('Mengkonversi halaman PDF ke gambar...')
page_images = pdf_to_images(PDF_PATH, START_PAGE, end_page, DPI)
print(f'{len(page_images)} gambar siap diproses.')
print()

print(f'Memuat model {MODEL_NAME}...')
processor, model = load_model(MODEL_NAME)
print()

all_results = []
for page_num, image in page_images:
    print(f'[Halaman {page_num}/{end_page}] Running VLM inference...')
    try:
        raw_output = run_inference(image, processor, model)
        data = extract_json(raw_output)
        rows, title, columns = save_page_output(data, page_num, output_dir)
        all_results.append({
            'page': page_num,
            'table_title': title,
            'columns': columns,
            'rows': rows,
        })
        print(f'[Halaman {page_num}] OK -- {len(rows)} baris diekstrak')
    except Exception as e:
        print(f'[Halaman {page_num}] ERROR: {e}')
        all_results.append({'page': page_num, 'error': str(e), 'rows': []})

print()
print('Menyimpan output gabungan...')
json_path, csv_path = save_combined_output(all_results, output_dir)

success = sum(1 for r in all_results if not r.get('error'))
failed = len(all_results) - success
print(f'Selesai! {success}/{len(all_results)} halaman berhasil, {failed} gagal.')
print(f'Output: {output_dir}/')

## Step 5 — Download Hasil

Jalankan cell ini untuk mendownload semua output sebagai file ZIP.

Isi ZIP:
- `page_XXX.json` / `page_XXX.csv` — hasil per halaman
- `combined.json` / `combined.csv` — semua halaman digabung

In [ ]:
import shutil
from google.colab import files as colab_files

print('Mengemas output...')
shutil.make_archive('vlm_output', 'zip', OUTPUT_DIR)
print('Mendownload vlm_output.zip ...')
colab_files.download('vlm_output.zip')